# 03 — Regression Modeling
**NBA Player Performance & Trade Value Predictor**  
DATA 300 · Spring 2026 · Anh Vu & Drew Norton

---

### Goals of this notebook
1. Load the feature dataset from notebook 02
2. Train and evaluate **4 models** per target: KNN, Linear Regression, Random Forest, Gradient Boosting
3. Use **5-fold cross-validation** on the training set for model selection
4. Compare **lag1 vs rolling features** to see which set works better
5. Select the best model per target and generate **next-season predictions**
6. Visualize feature importance and model comparison
7. Save predictions for notebook 05 (trade value score)

### Models
| Model | Role | Course connection |
|---|---|---|
| KNN Regressor | Baseline | Week 1–2 |
| Linear Regression + Ridge | Interpretable baseline | Week 4 |
| Random Forest | Ensemble, handles non-linearity | Week 13 |
| Gradient Boosting | Primary — best expected accuracy | Week 13 |


---
## 0. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import json
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

os.makedirs('figures', exist_ok=True)
os.makedirs('models',  exist_ok=True)
os.makedirs('data/processed', exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'sans-serif',
})

RANDOM_STATE = 42
print('Imports OK')

---
## 1. Load Data & Config

In [ ]:
df = pd.read_csv('data/processed/players_features.csv')

with open('data/processed/feature_config.json') as f:
    config = json.load(f)

FEATURES_LAG1    = config['FEATURES_LAG1']
FEATURES_ROLLING = config['FEATURES_ROLLING']
TARGET_COLS      = config['TARGET_COLS']
TARGETS          = config['TARGETS']
TRAIN_SEASONS    = config['TRAIN_SEASONS']
TEST_SEASONS     = config['TEST_SEASONS']

# Only keep features that exist in loaded data
FEATURES_LAG1    = [f for f in FEATURES_LAG1    if f in df.columns]
FEATURES_ROLLING = [f for f in FEATURES_ROLLING if f in df.columns]

df_train = df[df['season'].isin(TRAIN_SEASONS)].copy()
df_test  = df[df['season'].isin(TEST_SEASONS)].copy()

print(f'Loaded: {df.shape}')
print(f'Train: {len(df_train):,} | Test: {len(df_test):,}')
print(f'Targets: {TARGET_COLS}')
print(f'Lag1 features: {len(FEATURES_LAG1)} | Rolling features: {len(FEATURES_ROLLING)}')

---
## 2. Define Models

Each model is wrapped in a `Pipeline` with `StandardScaler`.  
This is essential for KNN and Ridge — distance/regularization are scale-sensitive.  
Tree-based models (RF, GBM) don't need scaling but it doesn't hurt.

In [ ]:
def make_models():
    """Returns a fresh dict of model pipelines."""
    return {
        'KNN': Pipeline([
            ('scaler', StandardScaler()),
            ('model',  KNeighborsRegressor(n_neighbors=10, weights='distance', metric='euclidean'))
        ]),
        'Linear Regression': Pipeline([
            ('scaler', StandardScaler()),
            ('model',  LinearRegression())
        ]),
        'Ridge': Pipeline([
            ('scaler', StandardScaler()),
            ('model',  Ridge(alpha=1.0))
        ]),
        'Random Forest': Pipeline([
            ('scaler', StandardScaler()),
            ('model',  RandomForestRegressor(
                n_estimators=200, max_depth=8,
                min_samples_leaf=5, random_state=RANDOM_STATE, n_jobs=-1
            ))
        ]),
        'Gradient Boosting': Pipeline([
            ('scaler', StandardScaler()),
            ('model',  GradientBoostingRegressor(
                n_estimators=200, learning_rate=0.05,
                max_depth=4, min_samples_leaf=5,
                subsample=0.8, random_state=RANDOM_STATE
            ))
        ]),
    }

MODEL_NAMES = list(make_models().keys())
print('Models defined:')
for name in MODEL_NAMES:
    print(f'  {name}')

---
## 3. Cross-Validation — Feature Set Comparison

First we compare lag1-only features vs rolling features using 5-fold CV on the training set.  
We test on `PER_next` as the primary target for this comparison.

**⏱ Expected time: ~2–3 minutes**

In [ ]:
from sklearn.model_selection import cross_validate

TARGET_FOR_COMPARISON = 'PER_next'
y_train_per = df_train[TARGET_FOR_COMPARISON].values

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

feature_comparison = []

for feat_name, feat_cols in [('Lag1 Only', FEATURES_LAG1), ('Rolling (3yr)', FEATURES_ROLLING)]:
    X_train = df_train[feat_cols].values
    
    for model_name in ['KNN', 'Linear Regression', 'Random Forest', 'Gradient Boosting']:
        model = make_models()[model_name]
        cv_results = cross_validate(
            model, X_train, y_train_per, cv=kf,
            scoring=['neg_root_mean_squared_error', 'neg_mean_absolute_error', 'r2'],
            return_train_score=False
        )
        feature_comparison.append({
            'feature_set': feat_name,
            'model':       model_name,
            'RMSE':        -cv_results['test_neg_root_mean_squared_error'].mean(),
            'RMSE_std':    cv_results['test_neg_root_mean_squared_error'].std(),
            'MAE':         -cv_results['test_neg_mean_absolute_error'].mean(),
            'R2':          cv_results['test_r2'].mean(),
        })
        print(f'{feat_name:18} | {model_name:22} | RMSE={-cv_results["test_neg_root_mean_squared_error"].mean():.3f} | R2={cv_results["test_r2"].mean():.3f}')

df_feat_compare = pd.DataFrame(feature_comparison)
print('\nFeature set comparison done.')

In [ ]:
# Pick the better feature set based on best Gradient Boosting RMSE
gb_lag1    = df_feat_compare[(df_feat_compare['model']=='Gradient Boosting') & (df_feat_compare['feature_set']=='Lag1 Only')]['RMSE'].values[0]
gb_rolling = df_feat_compare[(df_feat_compare['model']=='Gradient Boosting') & (df_feat_compare['feature_set']=='Rolling (3yr)')]['RMSE'].values[0]

BEST_FEATURES = FEATURES_ROLLING if gb_rolling <= gb_lag1 else FEATURES_LAG1
BEST_FEAT_NAME = 'Rolling (3yr)' if gb_rolling <= gb_lag1 else 'Lag1 Only'

print(f'Gradient Boosting RMSE — Lag1: {gb_lag1:.3f} | Rolling: {gb_rolling:.3f}')
print(f'→ Using: {BEST_FEAT_NAME} features for all models')

---
## 4. Full Cross-Validation — All Targets × All Models

5-fold CV on training set using the selected feature set.  
We evaluate all three targets (PER, PTS, WS) across all 5 models.

**⏱ Expected time: ~4–5 minutes**

In [ ]:
X_train = df_train[BEST_FEATURES].values
X_test  = df_test[BEST_FEATURES].values

cv_results_all = []

print(f'Running 5-fold CV with {BEST_FEAT_NAME} features...\n')

for target in TARGET_COLS:
    y_train = df_train[target].values
    print(f'Target: {target}')
    
    for model_name in MODEL_NAMES:
        model = make_models()[model_name]
        cv = cross_validate(
            model, X_train, y_train, cv=kf,
            scoring=['neg_root_mean_squared_error', 'neg_mean_absolute_error', 'r2'],
        )
        rmse = -cv['test_neg_root_mean_squared_error'].mean()
        mae  = -cv['test_neg_mean_absolute_error'].mean()
        r2   = cv['test_r2'].mean()
        
        cv_results_all.append({
            'target': target, 'model': model_name,
            'CV_RMSE': round(rmse, 4), 'CV_MAE': round(mae, 4), 'CV_R2': round(r2, 4),
            'CV_RMSE_std': round(-cv['test_neg_root_mean_squared_error'].std(), 4),
        })
        print(f'  {model_name:22} RMSE={rmse:.3f} ± {-cv["test_neg_root_mean_squared_error"].std():.3f}  MAE={mae:.3f}  R²={r2:.3f}')
    print()

df_cv = pd.DataFrame(cv_results_all)
print('CV complete.')

---
## 5. Model Comparison Visualization

In [ ]:
MODEL_COLORS = {
    'KNN':                '#888888',
    'Linear Regression':  '#4682B4',
    'Ridge':              '#5F9EA0',
    'Random Forest':      '#2E8B57',
    'Gradient Boosting':  '#E8A020',
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=False)

for ax, target in zip(axes, TARGET_COLS):
    target_df = df_cv[df_cv['target'] == target].sort_values('CV_RMSE')
    colors = [MODEL_COLORS.get(m, 'gray') for m in target_df['model']]
    
    bars = ax.barh(target_df['model'], target_df['CV_RMSE'],
                   color=colors, edgecolor='white', linewidth=0.5)
    ax.errorbar(
        target_df['CV_RMSE'], target_df['model'],
        xerr=target_df['CV_RMSE_std'],
        fmt='none', color='black', capsize=3, linewidth=1.2
    )
    
    # Annotate RMSE values
    for bar, rmse in zip(bars, target_df['CV_RMSE']):
        ax.text(rmse + 0.01, bar.get_y() + bar.get_height()/2,
                f'{rmse:.3f}', va='center', fontsize=8.5)
    
    ax.set_title(f'{target}\n5-Fold CV RMSE (lower = better)', fontsize=11, fontweight='bold')
    ax.set_xlabel('RMSE')
    ax.grid(axis='x', alpha=0.3)

plt.suptitle('Model Comparison — Cross-Validation RMSE by Target', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/model_comparison_rmse.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved → figures/model_comparison_rmse.png')

In [ ]:
# R² comparison — how much variance does each model explain?
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, target in zip(axes, TARGET_COLS):
    target_df = df_cv[df_cv['target'] == target].sort_values('CV_R2', ascending=True)
    colors = [MODEL_COLORS.get(m, 'gray') for m in target_df['model']]
    
    ax.barh(target_df['model'], target_df['CV_R2'], color=colors, edgecolor='white')
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlim(-0.1, 1.0)
    ax.set_title(f'{target}\n5-Fold CV R²', fontsize=11, fontweight='bold')
    ax.set_xlabel('R² Score')
    ax.grid(axis='x', alpha=0.3)
    
    for bar, r2 in zip(ax.patches, target_df['CV_R2']):
        ax.text(max(r2 + 0.01, 0.01), bar.get_y() + bar.get_height()/2,
                f'{r2:.3f}', va='center', fontsize=8.5)

plt.suptitle('Model Comparison — R² by Target', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/model_comparison_r2.png', bbox_inches='tight', dpi=150)
plt.show()

---
## 6. Select Best Model & Evaluate on Test Set

Select the model with the **lowest CV RMSE** per target, then evaluate on the held-out test set (2023–2024).  
This is the final, honest evaluation — not seen during training or model selection.

In [ ]:
import pickle

best_models  = {}   # best fitted model per target
test_results = []   # test set performance

print('Training best models on full training set and evaluating on test set...\n')

for target in TARGET_COLS:
    # Find best model from CV
    best_row = df_cv[df_cv['target'] == target].sort_values('CV_RMSE').iloc[0]
    best_name = best_row['model']
    
    # Retrain on full training set
    model = make_models()[best_name]
    y_train = df_train[target].values
    y_test  = df_test[target].values
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)
    
    best_models[target] = {'model': model, 'name': best_name}
    test_results.append({
        'target': target, 'best_model': best_name,
        'CV_RMSE': best_row['CV_RMSE'], 'Test_RMSE': round(rmse, 4),
        'Test_MAE': round(mae, 4), 'Test_R2': round(r2, 4)
    })
    
    # Save model to disk
    with open(f'models/best_{target}.pkl', 'wb') as f:
        pickle.dump(model, f)
    
    print(f'{target}')
    print(f'  Best model:  {best_name}')
    print(f'  CV RMSE:     {best_row["CV_RMSE"]:.4f}')
    print(f'  Test RMSE:   {rmse:.4f}  |  MAE: {mae:.4f}  |  R²: {r2:.4f}')
    print()

df_test_results = pd.DataFrame(test_results)
display(df_test_results)

---
## 7. Predicted vs Actual Plots

A good model's predictions should cluster tightly around the diagonal (y = x).  
Systematic deviations reveal where the model struggles (e.g., underestimating elite players).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, target in zip(axes, TARGET_COLS):
    y_test  = df_test[target].values
    y_pred  = best_models[target]['model'].predict(X_test)
    r2      = r2_score(y_test, y_pred)
    rmse    = np.sqrt(mean_squared_error(y_test, y_pred))
    
    ax.scatter(y_test, y_pred, alpha=0.45, s=18, color='#1F3864', edgecolors='white', linewidths=0.2)
    
    # Perfect prediction line
    lims = [min(y_test.min(), y_pred.min()) - 1, max(y_test.max(), y_pred.max()) + 1]
    ax.plot(lims, lims, 'r--', linewidth=1.5, alpha=0.7, label='Perfect prediction')
    
    ax.set_xlabel(f'Actual {target}', fontsize=11)
    ax.set_ylabel(f'Predicted {target}', fontsize=11)
    ax.set_title(f'{target}\n{best_models[target]["name"]}', fontsize=11, fontweight='bold')
    ax.text(0.05, 0.92, f'R² = {r2:.3f}\nRMSE = {rmse:.3f}',
            transform=ax.transAxes, fontsize=9,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))
    ax.legend(fontsize=8)
    ax.grid(alpha=0.2)

plt.suptitle('Predicted vs Actual — Test Set (2023–2024)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/predicted_vs_actual.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved → figures/predicted_vs_actual.png')

---
## 8. Feature Importance

For tree-based models, feature importance shows which input features drive predictions most.  
For Linear/Ridge, we use coefficient magnitude.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 6))

for ax, target in zip(axes, TARGET_COLS):
    model_info = best_models[target]
    model_name = model_info['name']
    pipeline   = model_info['model']
    inner      = pipeline.named_steps['model']
    
    # Get importances depending on model type
    if hasattr(inner, 'feature_importances_'):
        importances = inner.feature_importances_
    elif hasattr(inner, 'coef_'):
        importances = np.abs(inner.coef_)
    else:
        ax.set_title(f'{target}\n(importance N/A for KNN)')
        continue
    
    feat_imp = pd.Series(importances, index=BEST_FEATURES).nlargest(15)
    
    colors = ['#E8A020' if i == 0 else '#1F3864' for i in range(len(feat_imp))]
    ax.barh(feat_imp.index[::-1], feat_imp.values[::-1], color=colors[::-1], edgecolor='white')
    ax.set_title(f'{target}\n{model_name} — Top 15 Features', fontsize=10, fontweight='bold')
    ax.set_xlabel('Importance')
    ax.grid(axis='x', alpha=0.3)

plt.suptitle('Feature Importance by Target Variable', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/feature_importance.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved → figures/feature_importance.png')

---
## 9. Learning Curves (Bias–Variance)

Learning curves show how model performance changes with more training data.  
- **High training error + high validation error** = underfitting (high bias)  
- **Low training error + high validation error** = overfitting (high variance)  
- **Both converging** = good fit

In [ ]:
from sklearn.model_selection import learning_curve

# Plot learning curve for best model on PER_next
target_lc   = 'PER_next'
model_lc    = make_models()[best_models[target_lc]['name']]
y_train_lc  = df_train[target_lc].values

train_sizes, train_scores, val_scores = learning_curve(
    model_lc, X_train, y_train_lc,
    train_sizes=np.linspace(0.1, 1.0, 8),
    cv=5, scoring='neg_root_mean_squared_error',
    n_jobs=-1, random_state=RANDOM_STATE
)

train_rmse = -train_scores.mean(axis=1)
val_rmse   = -val_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
val_std    = val_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(train_sizes, train_rmse, 'o-', color='#1F3864', label='Training RMSE', linewidth=2)
ax.plot(train_sizes, val_rmse,   'o-', color='#E8A020', label='Validation RMSE', linewidth=2)
ax.fill_between(train_sizes, train_rmse - train_std, train_rmse + train_std, alpha=0.15, color='#1F3864')
ax.fill_between(train_sizes, val_rmse - val_std, val_rmse + val_std, alpha=0.15, color='#E8A020')
ax.set_xlabel('Training Set Size', fontsize=12)
ax.set_ylabel('RMSE', fontsize=12)
ax.set_title(f'Learning Curve — {best_models[target_lc]["name"]} on {target_lc}', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('figures/learning_curve.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved → figures/learning_curve.png')

---
## 10. Generate Predictions for All Players

Predict next-season stats for all players in the **most recent season** (2024).  
These predictions feed directly into notebook 05 (trade value score).

In [ ]:
# Use 2024 as the "current" season to predict 2025
df_latest = df[df['season'] == 2024].copy()

# Fill any missing features with training set medians
train_medians = df_train[BEST_FEATURES].median()
for col in BEST_FEATURES:
    if col in df_latest.columns:
        df_latest[col] = df_latest[col].fillna(train_medians[col])
    else:
        df_latest[col] = train_medians[col]

X_latest = df_latest[BEST_FEATURES].values

# Generate predictions
for target in TARGET_COLS:
    col_name = target.replace('_next', '_predicted')
    df_latest[col_name] = best_models[target]['model'].predict(X_latest)

# Show top predicted players
pred_cols = ['Player', 'Tm', 'Age', 'salary', 'salary_pct_cap'] + \
            [t.replace('_next', '_predicted') for t in TARGET_COLS]
pred_cols = [c for c in pred_cols if c in df_latest.columns]

df_predictions = df_latest[pred_cols].sort_values('PER_predicted', ascending=False)

print('Top 15 predicted PER for next season:')
display(df_predictions.head(15).reset_index(drop=True))

---
## 11. Save Results

In [ ]:
# Save predictions
df_predictions.to_csv('data/processed/player_predictions.csv', index=False)

# Save CV results table
df_cv.to_csv('data/processed/cv_results.csv', index=False)

# Save test set results
df_test_results.to_csv('data/processed/test_results.csv', index=False)

print('Saved files:')
print('  data/processed/player_predictions.csv  ← used in notebook 05')
print('  data/processed/cv_results.csv          ← used in report')
print('  data/processed/test_results.csv        ← used in report')
print('  models/best_*.pkl                      ← fitted model objects')
print('\n--- FINAL EVALUATION SUMMARY ---')
display(df_test_results[['target', 'best_model', 'CV_RMSE', 'Test_RMSE', 'Test_MAE', 'Test_R2']])
print('\n--- READY FOR NOTEBOOK 05: TRADE VALUE SCORE ---')

---
## Summary

| Step | What we did |
|---|---|
| Feature set comparison | Tested lag1-only vs rolling features; selected best |
| 5-fold CV | Evaluated KNN, Linear, Ridge, RF, GBM across all 3 targets |
| Model selection | Best model per target by CV RMSE |
| Test set evaluation | Final honest evaluation on 2023–2024 holdout |
| Visualizations | RMSE comparison, R² comparison, predicted vs actual, feature importance, learning curve |
| Predictions | Next-season PER, PTS, WS for all 2024 players |

**Next:** `05_trade_value.ipynb` — combine predictions with salary and age to build the trade value leaderboard.